[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/13DU-wor-cnM_bFFKDakqHvkOktzchg8_)

# Семинар 1 · Знакомство с временными рядами

1. Время в pandas
2. Загрузка готового ряда
3. Визуализация
4. Пропуски
5. Компоненты ряда
6. Train/test
7. Метрики
8. Бейзлайны и сравнение


In [ ]:
%pip install numpy pandas matplotlib plotly statsmodels sktime utilsforecast

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
pd.options.display.float_format = "{:.3f}".format
np.random.seed(42)

In [ ]:
import plotly.io as pio

pio.templates["transparent_lines"] = go.layout.Template(
    data={"scatter": [go.Scatter(opacity=0.7)]}
)

pio.templates.default = "plotly+transparent_lines"

## 1. Время в pandas

Рассмотрим таблицу месячных продаж: даты, порядок наблюдений и календарь


In [ ]:
toy_table = pd.DataFrame({"month": ["2025-04", "2025-01", "2025-02", "2025-05", "2025-06"],
                          "sales": [140, 100, 120, 150, 160]})
toy_table

`2025-04` выглядит как месяц, но пока это текст. Преобразуем всю колонку через `pd.to_datetime` и сравним тип до и после


In [ ]:
# format задает порядок полей, первое число служит меткой всего месяца
print("До:", toy_table["month"].dtype)
toy_table["month"] = pd.to_datetime(toy_table["month"], format="%Y-%m")
print("После:", toy_table["month"].dtype)

In [ ]:
# .dt открывает календарные свойства колонки дат
toy_table.assign(month_number=toy_table["month"].dt.month)

Перенесем дату в индекс и отсортируем наблюдения. Теперь период можно выбирать по календарю


In [ ]:
toy = toy_table.set_index("month")["sales"].sort_index()
toy

In [ ]:
toy.loc["2025-02":"2025-04"]

In [ ]:
# Как еще можно делать срезы
toy.loc["2025"]             # все наблюдения за 2025 год
toy.loc["2025-04"]          # все наблюдения за апрель
toy.loc[:"2025-04"]         # январь, февраль и апрель
toy.loc["2025-04":]         # апрель, май и июнь

Ожидаем одно наблюдение на месяц. Построим календарь и сравним его с фактическими датами


In [ ]:
# MS — начала месяцев
toy_calendar = pd.date_range(toy.index.min(), toy.index.max(), freq="MS")
toy_calendar.difference(toy.index)

**Шпаргалка: что можно передать в `freq`**

`freq` задает шаг и, для календарных частот, правило выбора дат. Например, `MS` — не «каждые 30 дней», а «первое число каждого месяца».

| `freq` | Какие даты или моменты получим |
|---|---|
| `ns`, `us`, `ms` | Наносекунды, микросекунды, миллисекунды |
| `s`, `min`, `h` | Секунды, минуты, часы |
| `bh`, `cbh` | Рабочие часы; `cbh` — с настраиваемым календарем |
| `D` | Каждый календарный день |
| `B`, `C` | Будние дни; `C` — с настраиваемым календарем |
| `W`, `W-SUN` | Каждое воскресенье (`W` по умолчанию означает `W-SUN`) |
| `W-MON`, `W-TUE`, `W-WED`, `W-THU`, `W-FRI`, `W-SAT` | Раз в неделю в указанный день |
| `MS`, `ME` | Первое и последнее число каждого месяца соответственно |
| `SMS`, `SME` | Дважды в месяц: 1-е и 15-е либо 15-е и последний день |
| `BMS`, `BME` | Первый и последний будний день месяца |
| `CBMS`, `CBME` | Первый и последний рабочий день месяца по настраиваемому календарю |
| `QS`, `QE` | Начала и окончания календарных кварталов |
| `BQS`, `BQE` | Первый и последний будний день квартала |
| `YS`, `YE` | Начала и окончания календарных лет |
| `BYS`, `BYE` | Первый и последний будний день года |

**Кратность:** `2h` — каждые два часа, `15min` — каждые 15 минут, `2MS` — начало каждого второго месяца.

**Границы:** если `start` не лежит на сетке, `date_range` начинает со следующей подходящей даты. Например, с `2025-01-15` при `MS` первая метка будет `2025-02-01`.

**Регистр важен:** `MS` — начало месяца, `ms` — миллисекунда. `B` по умолчанию пропускает субботу и воскресенье, но не учитывает государственные праздники. Для своего календаря можно передать объект `pd.offsets.CustomBusinessDay(...)`.

Здесь частоты для `date_range` и `resample`. У `Period` свои обозначения: например, `M` обозначает месяц как период. [Список частот и календарных сдвигов в документации pandas](https://pandas.pydata.org/docs/user_guide/timeseries.html#offset-aliases)


`shift(1)` сдвигает значения на одну **строку**

In [ ]:
pd.DataFrame({"Продажи": toy, "Предыдущая строка": toy.shift(1)})

In [ ]:
toy_monthly = toy.reindex(toy_calendar)
pd.DataFrame({
    "Продажи": toy_monthly,
    "Продажи за прошлый месяц": toy_monthly.shift(1)
    })

 **Агрегация по кварталам**

`resample` группирует по календарным интервалам. Суммируем продажи и рядом проверяем, сколько месяцев известно в каждом квартале


In [ ]:
toy_monthly.resample("QS").agg(["sum", "count"])

**Календарные сдвиги: `pd.DateOffset` и `pd.offsets`**

Позволяет **сдвинуть одну дату по календарному правилу**. Например, перейти на ближайший следующий будний день или на начало следующего месяца.


In [ ]:
friday = pd.Timestamp("2026-11-06")
print("Плюс один календарный день:", friday + pd.DateOffset(days=1))
print("Следующий будний день:", friday + pd.offsets.BusinessDay())
print("Следующее начало месяца:", friday + pd.offsets.MonthBegin())

In [ ]:
# days прибавляет дни, day заменяет число месяца
print("Прибавить день:", friday + pd.DateOffset(days=1))
print("Поставить первое число:", friday + pd.DateOffset(day=1))

In [ ]:
# freq принимает строковый код или соответствующий объект offset
start = "2025-01-15"
by_alias = pd.date_range(start, periods=3, freq="MS")
by_offset = pd.date_range(start, periods=3, freq=pd.offsets.MonthBegin())
by_alias, by_alias.equals(by_offset)

## 2. Загрузка готового ряда

Данные — месячные продажи магазинов одежды и аксессуаров в США, млн долларов, без сезонной корректировки

Источник — U.S. Census Bureau через [FRED](https://fred.stlouisfed.org/series/MRTSSM448USN)

Используем январь 2012 — август 2026

In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MRTSSM448USN"
raw = pd.read_csv(url)
raw.head()

In [ ]:
raw.observation_date.min(), raw.observation_date.max()

In [ ]:
raw.dtypes

Тот же CSV можно прочитать с распознаванием дат


In [ ]:
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MRTSSM448USN"
parsed = pd.read_csv(url, parse_dates=["observation_date"])
parsed.dtypes

In [ ]:
df = parsed.set_index("observation_date").sort_index()
df = df.loc["2012-01":"2026-08"]

df = df["MRTSSM448USN"].rename("sales")
df.head()

In [ ]:
df.index.is_monotonic_increasing

In [ ]:
# Проверяем границы, уникальность и частоту
df.index.min(), df.index.max(), df.index.is_unique, pd.infer_freq(df.index)

## 3. Визуализация


### Matplotlib


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df)
ax.set(title="Продажи магазинов одежды в США", xlabel="Дата", ylabel="Продажи, млн USD")
fig.tight_layout()
plt.show()

### Plotly


In [ ]:
fig = px.line(df.to_frame(), x=df.index, y="sales",
              title="Продажи магазинов одежды в США",
              labels={"sales": "Млн USD", "observation_date": "Дата"})
fig.show()

__`Вопрос:`__ **Есть ли тренд, сезонность, необычные участки?**


In [ ]:
# Сравниваем одинаковые месяцы разных лет, чтобы увидеть повторяющиеся пики
recent = df.loc["2021":"2025"].to_frame()
recent["year"], recent["month"] = recent.index.year, recent.index.month
fig = px.line(recent, x="month", y="sales", color="year", markers=True,
              title="Сравнение сезонного рисунка по годам")
fig.show()

## 4. Пропуски

Проверяем неизвестные значения и отсутствующие даты


In [ ]:
# NaN в значениях и дубликаты временных меток — разные проблемы
df.isna().sum(), df.index.duplicated().sum()

In [ ]:
# Строим ожидаемый календарь, difference находит отсутствующие строки
calendar = pd.date_range(df.index.min(), df.index.max(), freq="MS")
calendar.difference(df.index)

В исходной истории пропусков нет. Удалим одну дату и заменим еще одно значение на NaN, работаем только с копией


In [ ]:
with_gaps = df.drop(pd.Timestamp("2023-06-01")).copy()
with_gaps.loc["2023-05-01"] = np.nan
with_gaps.loc["2023-01":"2023-12"]

In [ ]:
# isna видит неизвестное значение, но не удаленную строку, поэтому нужны обе проверки
with_gaps.isna().sum(), calendar.difference(with_gaps.index)

In [ ]:
# reindex восстанавливает сетку, теперь оба пропуска представлены как NaN
aligned = with_gaps.reindex(calendar)
aligned[aligned.isna()]

In [ ]:
comparison = pd.DataFrame({"Исходный ряд": df, "С пропусками": aligned})
fig = px.line(comparison.loc["2023-01":"2023-12"], markers=True,
              title="Удаленные наблюдения", labels={"value": "Млн USD"})
fig.update_traces()
fig.show()

In [ ]:
filled = pd.DataFrame({"Факт": df, "ffill": aligned.ffill(),
                       "Интерполяция": aligned.interpolate(method="time"),
                       "Прошлый год": aligned.fillna(aligned.shift(12))})
fig = px.line(filled.loc["2023-01":"2023-12"], markers=True,
              title="Сравнение заполнения пропусков", labels={"value": "Млн USD"})
fig.show()


## 5. Компоненты временного ряда

### Аддитивный пример

$$y_t=T_t+S_t+R_t$$

$$T_t=100+3t,\quad S_t=100\sin\left(\frac{2\pi t}{m}\right),\quad R_t \sim N(0, 2^2),\qquad m = 12$$

t — номер наблюдения от 0

m — число наблюдений в одном сезоне

T — тренд, S — сезонность, R — остаток

In [ ]:
T = 12 * 10 # 10 лет
demo_dates = pd.date_range("2015-01", periods=T, freq="MS")
position = np.arange(T)
period = 12

In [ ]:
trend = 100 + 3 * position
season = 100 * np.sin(2 * np.pi * position / period)
residual = np.random.normal(loc=0, scale=2, size=len(position))

In [ ]:
additive = pd.DataFrame({"Тренд": trend, "Сезонность": season,
                         "Остаток": residual}, index=demo_dates)
additive["Ряд"] = additive.sum(axis=1)
additive

In [ ]:
def show_components(data, title):
    order = ["Ряд", "Тренд", "Сезонность", "Остаток"]
    long = data.rename_axis("Дата").reset_index().melt(id_vars="Дата", var_name="Компонента")
    fig = px.line(long, x="Дата", y="value", facet_row="Компонента", height=750,
                  category_orders={"Компонента": order}, title=title)
    fig.update_yaxes(matches=None, title_text="")
    fig.for_each_annotation(lambda label: label.update(text=label.text.split("=")[-1]))
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
show_components(additive, "Аддитивный синтетический ряд")

### Мультипликативный пример

$$y_t=T_t\times S_t\times R_t$$

$$T_t = 100 + 3t,\quad S_t = 1 + 0.5\sin\left(\frac{2\pi t}{m}\right),\quad R_t \sim U(0.95, 1.05)$$



In [ ]:
trend = 100 + 3 * position
season_factor = 1 + 0.5 * np.sin(2 * np.pi * position / period)

residual_factor = np.random.uniform(0.95, 1.05, size=len(position)) # равномерное расперделение

multiplicative = pd.DataFrame({
    "Тренд": trend,
    "Сезонность": season_factor,
    "Остаток": residual_factor,
}, index=demo_dates)

multiplicative["Ряд"] = trend * season_factor * residual_factor
multiplicative

In [ ]:
show_components(multiplicative, "Мультипликативный синтетический ряд")

### Компоненты реального ряда

Сравним результат готового разложения с учебными примерами

Сам алгоритм разложения рассмотрим позднее


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

dec_mul = seasonal_decompose(df, model="multiplicative", period=12)

In [ ]:
real_components = pd.DataFrame({"Ряд": df, "Тренд": dec_mul.trend,
                                "Сезонность": dec_mul.seasonal,
                                "Остаток": dec_mul.resid})
show_components(real_components, "Разложение ряда по компонентам ")

In [ ]:
dec_add = seasonal_decompose(df, model="additive", period=12)
real_components = pd.DataFrame({"Ряд": df, "Тренд": dec_add.trend,
                                "Сезонность": dec_add.seasonal,
                                "Остаток": dec_add.resid})
show_components(real_components, "Разложение ряда по компонентам ")

In [ ]:
errors = pd.DataFrame({
    "Аддитивное": df - (dec_add.trend + dec_add.seasonal),
    "Мультипликативное": df - (dec_mul.trend * dec_mul.seasonal),
})

px.line(errors, title="Что осталось после учета тренда и сезонности",
        labels={"value": "Отклонение, млн USD"}).show()

## 6. Train/test

Главное правило: при прогнозе будущего используем только информацию, доступную на дату построения прогноза


In [ ]:
# Так делать нельзя!
from sklearn.model_selection import train_test_split

random_train, random_test = train_test_split(df, test_size=12*3, shuffle=True, random_state=42)
random_train.index.max(), random_test.index.min()

In [ ]:
random_split = pd.concat([random_train.rename("Случайный train"),
                          random_test.rename("Случайный test")], axis=1).sort_index()
long_split = random_split.rename_axis("Дата").reset_index().melt(id_vars="Дата")
fig = px.scatter(long_split, x="Дата", y="value", color="variable",
                 title="Случайное разбиение", labels={"value": "Млн USD"})
fig.show()


In [ ]:
# df = df.sort_index() - у нас уже отсортирован
train, test = train_test_split(df, test_size=12*3, shuffle=False)
train.index.max(), test.index.min()

In [ ]:
split = pd.concat([train.rename("Train"), test.rename("Test")], axis=1)
fig = px.line(split, title="Разбиение train/test", labels={"value": "Млн USD"})
fig.add_vline(x=test.index[0].isoformat())
fig.show()

## 7. Метрики

Напишем WAPE самостоятельно

$$\mathrm{WAPE}=100\%\cdot\frac{\sum_{i=1}^{H}|y_i-\hat y_i|}{\sum_{i=1}^{H}|y_i|}$$

Для нулевого знаменателя договоримся возвращать NaN

Остальные метрики возьмем готовыми из sktime


In [ ]:
def wape(actual, prediction):
    actual, prediction = np.asarray(actual), np.asarray(prediction)
    denominator = np.abs(actual).sum()
    if denominator == 0:
        return np.nan
    return 100 * np.abs(actual - prediction).sum() / denominator

Посмотрим на примере:

| Факт | Прогноз A | Прогноз B |
|---:|---:|---:|
|100 | 90 | 100 |
|10 | 10 | 5 |

In [ ]:
actual = np.array([100., 10.])
example_a = np.array([90., 10.])
example_b = np.array([100., 5.])

# У какой модели ошибка будет меньше?
wape(actual, example_a), wape(actual, example_b)

Сравним с библиотечной реализацией

In [ ]:
# !pip install utilsforecast

In [ ]:
from utilsforecast.losses import wape as wape_lib

comparison = pd.DataFrame({
    "unique_id": "toy",
    "y": actual,
    "A": example_a,
    "B": example_b,
})

result = wape_lib(comparison, models=["A", "B"])
result[["A", "B"]]

In [ ]:
# !pip install sktime

In [ ]:
# mae, mape, mse, rmse можно было взять из sklearn

from sktime.performance_metrics.forecasting import mean_absolute_error as mae
from sktime.performance_metrics.forecasting import mean_squared_error as mse
from sktime.performance_metrics.forecasting import mean_absolute_percentage_error as mape
from sktime.performance_metrics.forecasting import mean_absolute_scaled_error as mase
from sktime.performance_metrics.forecasting import relative_loss
from functools import partial


In [ ]:
rmse = partial(mse, square_root=True)
smape = partial(mape, symmetric=True)

mae(actual, example_a), mse(actual, example_a), rmse(actual, example_a)

In [ ]:
pd.DataFrame({"A": [100 * mape(actual, example_a), wape(actual, example_a), smape(actual, example_a)],
              "B": [100 * mape(actual, example_b), wape(actual, example_b), smape(actual, example_b)]},
             index=["MAPE, %", "WAPE, %", "sMAPE, %"])

## 8. Бейзлайны и сравнение

Каждый метод реализуем сами и отдельно посмотрим на его прогноз

Все получают только train, горизонт задаем заранее, факты test используются только для графика и оценки


In [ ]:
# Создаем будущий календарь для прогнозов
forecast_dates = test.index
predictions = pd.DataFrame(index=forecast_dates)

In [ ]:
def forecast_frame(prediction, history_prediction):
    return pd.DataFrame({"Train: факт": train, "Test: факт": test,
                         "Train: прогноз на 1 шаг": history_prediction,
                         "Test: прогноз на 12 шагов": pd.Series(prediction, index=forecast_dates)})

In [ ]:
# Создаем таблицу для исторических прогнозов
train_predictions = pd.DataFrame(index=train.index)

Для каждой модели показываем факты и прогнозы на train и test

**На train:** прогноз на один шаг с обновлением истории фактическими значениями

**На test:** прогноз на весь горизонт из конца train без новых фактов

Вертикальная линия — начало test, метрики считаем по test

В начале истории прогноз отсутствует, если еще не хватает наблюдений


In [ ]:
def show_forecast(prediction, title, history_prediction):
    frame = forecast_frame(prediction, history_prediction)
    fig = px.line(
        frame, title=title,
        labels={"value": "Млн USD", "index": "Дата"},
        color_discrete_sequence=["#8D9BAD", "#8D9BAD", "#27867D", "#B76B2C"],
    )
    fig.add_vline(x=test.index[0].isoformat())
    fig.show()

### 8.1. Среднеарифметический метод

$$\hat y_{T+h\mid T}=\frac{1}{T}\sum_{t=1}^{T}y_t$$

In [ ]:
def mean_forecast(history, horizon):
    return np.full(horizon, np.mean(history))

In [ ]:
name = "Среднеарифметический метод"
predictions[name] = mean_forecast(train, len(test))
train_predictions[name] = train.shift(1).expanding().mean()
show_forecast(predictions[name], name, train_predictions[name])

### 8.2. Наивный прогноз

$$\hat y_{T+h\mid T}=y_T$$

In [ ]:
def naive_forecast(history, horizon):
    return np.full(horizon, np.asarray(history)[-1])

In [ ]:
name = "Наивный прогноз"
predictions[name] = naive_forecast(train, len(test))
train_predictions[name] = train.shift(1)
show_forecast(predictions[name], name, train_predictions[name])

### 8.3. Сезонный наивный прогноз

$$\hat y_{T+h\mid T}=y_{T-m+1+((h-1)\bmod m)}$$

In [ ]:
def seasonal_naive_forecast(history, horizon, period):
    values = np.asarray(history)
    if not 1 <= period <= len(values):
        raise ValueError("В истории должен помещаться сезон")
    return np.resize(values[-period:], horizon)

In [ ]:
name = "Сезонный наивный прогноз"
predictions[name] = seasonal_naive_forecast(train, len(test), period)
train_predictions[name] = train.shift(12)
show_forecast(predictions[name], name, train_predictions[name])

### 8.4. Прогноз с дрейфом

$$\hat y_{T+h\mid T}=y_T+h\frac{y_T-y_1}{T-1}$$

In [ ]:
def drift_forecast(history, horizon):
    values = np.asarray(history)
    if len(values) < 2:
        raise ValueError("Нужны хотя бы две точки")
    slope = (values[-1] - values[0]) / (len(values) - 1)
    return values[-1] + np.arange(1, horizon + 1) * slope

In [ ]:
def drift_on_train(train):
    """Строит прогноз с дрейфом на один шаг для каждой даты train"""
    prediction = pd.Series(index=train.index, dtype=float)

    for position in range(2, len(train)):
        history = train.iloc[:position]  # Все факты до текущей даты
        prediction.iloc[position] = drift_forecast(history, horizon=1)[0]

    return prediction

In [ ]:
name = "Прогноз с дрейфом"
predictions[name] = drift_forecast(train, len(test))
train_predictions[name] = drift_on_train(train)
show_forecast(predictions[name], name, train_predictions[name])

**Вопрос:** Почему в знаменателе $T-1$, а не $T$?


### 8.5. Скользящее среднее: фиксированное окно

$$\hat y_{T+h\mid T}=\frac{1}{k}\sum_{j=0}^{k-1}y_{T-j}$$


In [ ]:
def moving_mean_forecast(history, horizon, window):
    values = np.asarray(history)
    if not 1 <= window <= len(values):
        raise ValueError("Окно должно помещаться в историю")
    return np.full(horizon, values[-window:].mean())

In [ ]:
name = "Скользящее среднее: фиксированное окно"
window = 12

predictions[name] = moving_mean_forecast(train, len(test), window)
train_predictions[name] = train.shift(1).rolling(window).mean()
show_forecast(predictions[name], name, train_predictions[name])

__`Вопрос:`__ **Что будет если поставить window=1?**


### 8.6. Скользящее среднее: рекурсивный прогноз

$$\hat y_{T+h\mid T}=\frac{1}{k}\sum_{j=1}^{k}\tilde y_{T+h-j},\qquad
\tilde y_u=\begin{cases}y_u,&u\leq T\\\hat y_{u\mid T},&u>T\end{cases}$$

In [ ]:
def recursive_mean_forecast(history, horizon, window):
    available = list(history)
    if not 1 <= window <= len(available):
        raise ValueError("Окно должно помещаться в историю")
    result = []
    for step in range(horizon):
        prediction = np.mean(available[-window:])
        result.append(prediction)
        available.append(prediction)
    return np.asarray(result)

In [ ]:
name = "Рекурсивное среднее"
window = 12

predictions[name] = recursive_mean_forecast(train, len(test), window)
train_predictions[name] = train.shift(1).rolling(window).mean()
show_forecast(predictions[name], name, train_predictions[name])


**Вопрос:** К какому числу будет стремиться прогноз?


In [ ]:
small_history = [10., 20., 30.]
pd.DataFrame({"Без пересчета": moving_mean_forecast(small_history, 8, 3),
              "Рекурсивно": recursive_mean_forecast(small_history, 8, 3)})


### Сравним все модели в таблице метрик


In [ ]:
def basic_scores(actual, prediction):
    return {"MAE": mae(actual, prediction), "MSE": mse(actual, prediction),
            "RMSE": rmse(actual, prediction), "MAPE": 100 * mape(actual, prediction),
            "sMAPE": 100 * smape(actual, prediction), "WAPE": wape(actual, prediction)}


In [ ]:
scores = pd.DataFrame({name: basic_scores(test.to_numpy(), predictions[name].to_numpy())
                       for name in predictions}).T
scores.sort_values("MAE")


In [ ]:
benchmark = predictions["Наивный прогноз"].to_numpy()
for name in predictions:
    prediction = predictions[name].to_numpy()
    scores.loc[name, "MASE"] = mase(test.to_numpy(), prediction, y_train=train.to_numpy(), sp=period)
    scores.loc[name, "RelMAE"] = relative_loss(test.to_numpy(), prediction, y_pred_benchmark=benchmark)
    scores.loc[name, "RelRMSE"] = relative_loss(test.to_numpy(), prediction,
                                              y_pred_benchmark=benchmark, relative_loss_function=rmse)
scores.sort_values("MAE")

In [ ]:
fig = px.line(predictions, title="Все прогнозы: сравните форму и уровень")
fig.add_trace(go.Scatter(x=test.index, y=test, name="Факт"))
fig.show()